In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import pandas as pd

In [2]:
def extract_data(file_path):
    items = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i,line in enumerate(f):
            items.append(item(i, line))
    return items

In [3]:
# Function to load 1D bin packing instances from files
def load_binpack_file(filename, instance_idx=0, is_float=False):
    with open(filename, 'r') as f:
        lines = f.readlines()
    line_idx = 0
    P = int(lines[line_idx].strip())
    line_idx += 1
    for p in range(instance_idx + 1):
        identifier = lines[line_idx].strip()
        line_idx += 1
        parts = lines[line_idx].strip().split()
        capacity = float(parts[0]) if is_float else int(parts[0])
        n = int(parts[1])
        best_known = int(parts[2])
        line_idx += 1
        sizes = []
        for _ in range(n):
            if is_float:
                sizes.append(float(lines[line_idx].strip()))
            else:
                sizes.append(int(lines[line_idx].strip()))
            line_idx += 1
    return sizes, capacity, best_known, identifier

In [4]:
class item:

    def __init__(self, idx, size):
        self.id = int(idx)
        self.size = int(size)

    def get_id(self):
        return self.id

class bin_c:

    def __init__(self, c):
        self.capacity = c
        self.load = 0
        self.items = []

    def _can_fit(self, item):
        return (self.load + item.size) <= self.capacity
    
    def add_item(self, item):
        if self._can_fit(item):
            self.items.append(item)
            self.load += item.size
            return True
        return False

    def calc_phero(self, item, pheromone_table):
        if not self.items:
            return 1
        pheromone_vals = [] # Empty list for all pheromone vals
        for item_j in self.items:
            idx_j = item_j.get_id()
            pheromone_vals.append(pheromone_table[item.id][idx_j])
        return sum(pheromone_vals) / len(self.items)

In [5]:
class ACO:

    def __init__(self, items, c, beta, evaporation, seed=42):
        self.items = items
        self.n_items = len(items)
        self.num_ants = self.n_items
        self.max_iter = self.n_items
        self.capacity = c
        self.pheromones = np.ones((self.n_items, self.n_items)) * 0.01
        self.heur_import = beta
        self.evaporation = evaporation
        self.rng = np.random.default_rng(seed)
        self.best_solution = None
        self.best_quality = float('inf')

    def compute_nominator(self, item, box):
        if box._can_fit(item):
            heur_val = item.size
            phero_val = box.calc_phero(item, self.pheromones)
            return (phero_val * (heur_val ** self.heur_import))
        else:
            return 0
            
    def construct_solution(self, items):
        bins = [bin_c(self.capacity)] 
        unassigned_items = items.copy()

        while unassigned_items:
            for box in bins:
                prob = [] # List to hold the probabilities of all items
                for item in unassigned_items: # Iterate over all possible items
                    prob.append(self.compute_nominator(item, box)) # Get a list of probabilities of all items
                for i, item in enumerate(unassigned_items):
                    if box._can_fit(item):
                        #print(i)
                        numerator = prob[i]
                        denominator = sum(prob)
                        prob[i] = numerator / denominator
                    else:
                        continue
                # Normalize probabilities if possible, else put item in new bin
                total_prob = sum(prob)
                if total_prob == 0:  # no item can't fit in current bin
                    bins.append(bin_c(self.capacity))
                else:
                    normalized_probs = [p / total_prob for p in prob]
                    chosen_item = self.rng.choice(unassigned_items, p=normalized_probs, size=1)[0]
                    box.add_item(chosen_item)
                    unassigned_items.remove(chosen_item)

        return bins

    def update_pheromones(self, solutions):
        # Apply evaporation to all entries
        self.pheromones *= (1 - self.evaporation)

        # Update pheromones based on solutions quality
        for bins in solutions:
            bin_weights = sum([box.load for box in bins])
            quality = (bin_weights / self.capacity)**2 / len(bins) # better solution = more pheromone
            quality > self.best_quality
            
            for box in bins:
                for i in range(len(box.items)):
                    for j in range(i + 1, len(box.items)):
                        id_i = box.items[i].id
                        id_j = box.items[j].id
                        # Increase pheromone on pairs placed together
                        self.pheromones[id_i][id_j] += quality
                        self.pheromones[id_j][id_i] += quality

    def run(self):
        iteration = 0
        while iteration < self.max_iter: 
            solutions = []
            iteration += 1
            for i in range(self.num_ants):
                solutions.append(self.construct_solution(self.items))
                self.update_pheromones(solutions)
        return self.best_solution, self.best_quality

In [6]:
file_path = "data/test_data.txt"

item_list = extract_data(file_path)

In [ ]:
algo = ACO(item_list, 150, 0.5, 0.1, seed=42)

algo.run()